# Data Loading and Sampling

Goal of this notebook:
- load the HotpotQA validation split
- inspect the dataset structure
- create a fixed evaluation sample for all experiments
- save the sampled questions for reproducibility

Current objective:
- use a fixed sample of 500 questions
- keep the same sample across all configurations

In [1]:
from datasets import load_dataset
import pandas as pd
import json
from pathlib import Path

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset

dataset = load_dataset("hotpot_qa", "distractor", split="validation")
print(dataset)
print("Number of examples:", len(dataset))
print("Columns:", dataset.column_names)

Generating validation split: 100%|█| 7405/7405 [00:01<00:00, 4443.22 examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 7405
})
Number of examples: 7405
Columns: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


In [3]:
# We inspect the structure of the "context" field for one HotpotQA example to understand the data structure
# This is important before building the corpus, because errors can happen if we assume the wrong data structure.
example = dataset[0]

print("ID:", example["id"])
print("Question:", example["question"])
print("Answer:", example["answer"])
print("Type:", example["type"])
print("Level:", example["level"])

print("\nSupporting facts:")
print(example["supporting_facts"])

print("\nContext:")
print(example["context"])

ID: 5a8b57f25542995d1e6f1371
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Type: comparison
Level: hard

Supporting facts:
{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}

Context:
{'title': ['Ed Wood (film)', 'Scott Derrickson', 'Woodson, Arkansas', 'Tyler Bates', 'Ed Wood', 'Deliver Us from Evil (2014 film)', 'Adam Collis', 'Sinister (film)', 'Conrad Brooks', 'Doctor Strange (2016 film)'], 'sentences': [['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.', " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.", ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.'], ['Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.', ' He lives 

Observation:
The context is not a list of (title, sentences) tuples.
Instead, it is a dictionary with two parallel lists:
 - context["title"] contains document titles
 - context["sentences"] contains the corresponding list of sentences for each title

In [4]:
# We print the first two documents to verify that both lists align by position.
# This means:
# context["title"][i] belongs to context["sentences"][i]
context = example["context"]

print("Number of titles:", len(context["title"]))
print("Number of sentence lists:", len(context["sentences"]))

print("\nFirst 2 documents:\n")

for i, (title, sentences) in enumerate(zip(context["title"], context["sentences"])):
    print(f"Document {i+1}")
    print("Title:", title)
    print("Sentences:", sentences)
    print("-" * 60)
    
    if i == 1:   # stop after first 2 documents
        break

Number of titles: 10
Number of sentence lists: 10

First 2 documents:

Document 1
Title: Ed Wood (film)
Sentences: ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.', " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.", ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.']
------------------------------------------------------------
Document 2
Title: Scott Derrickson
Sentences: ['Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.', ' He lives in Los Angeles, California.', ' He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."']
----

Conclusion / observations:
 - Both lists have length 10 in this example, so the structure is aligned correctly.
 - Each title corresponds to one list of sentences.
 - This tells us that later, when building the corpus, we must iterate with:
       zip(context["title"], context["sentences"])
   and NOT with:
       for title, sentences in context
   because that would incorrectly iterate over dictionary keys.

Why this helps:
 - It gives us the correct pattern for transforming HotpotQA contexts into documents.


In [7]:
# We define a central output directory for all generated data in the project.
# This ensures that all intermediate data (samples, corpus, embeddings, etc.) are stored in a consistent and reusable location.

from pathlib import Path

output_dir = Path("/home/a/arfaoui/rag_project/data")

# Create the directory if it does not exist
output_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir.resolve())

Output directory: /home/a/arfaoui/rag_project/data


In [9]:
# We create a fixed random sample of questions from the validation set.
# Using a fixed seed ensures reproducibility:
# every time we rerun this code, we get the same 500 questions.
# Crucial for evaluation
sample_size = 500
# We fix a random seed to ensure reproducibility.
# This guarantees that the same 500 questions are selected every time.
seed = 250

sampled_dataset = dataset.shuffle(seed=seed).select(range(sample_size))
# Print to check
print("Sample size:", len(sampled_dataset))
print("First sampled question ID:", sampled_dataset[0]["id"])
print("First sampled question:", sampled_dataset[0]["question"])

# We now save the sampled dataset to disk.
# Later, DO NOT resample again, directly load form disk

sample_path = output_dir / "hotpotqa_sample_500.json"

sampled_dataset.to_json(sample_path)

print("Sample saved to:", sample_path)

# Why this matters:
# - All future configurations should be evaluated on the exact same question set.
# - This makes comparisons fair across chunk sizes and top-k values.
# - Otherwise, differences in F1 could come from different sampled questions
#   rather than from the retrieval settings themselves.

Sample size: 500
First sampled question ID: 5a8aeadc55429950cd6afbe0
First sampled question: What date in 2010 was a South Korean film starring  Kim Hyang-gi released?


Creating json from Arrow format: 100%|████████████| 1/1 [00:00<00:00,  1.58ba/s]

Sample saved to: /home/a/arfaoui/rag_project/data/hotpotqa_sample_500.json
